In [ ]:
using ProgressBars
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

# Emission spectrum for a linear oscillator

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
ω_sys=5*2*pi  # GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
L = C_inv / ω_sys^2 # L in 1/GHz
Z_sys = sqrt(L*C_inv) # Characteristic impedance (dimensionless)
κ=1/20; # 1/ns from Jeffrey et al.

In [ ]:
n_huge=2000
AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=90))
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
bath = Exp_decay
bath_params=(ω_c=50, ω_cut_off=100)
ϕ,q,t= One_d_linear(n, bath, bath_params, t_max; ω_sys=5*2pi, κ=1/20, C_inv=0.147*2*pi, β=∞, energy=false)

In [ ]:
plot(ϕ,q)

In [ ]:
ϕ_t_normal_modes,q_t_normal_modes,_ = One_d_linear(n, bath, bath_params, t_max; ω_sys=5*2pi, κ=1/20, C_inv=0.147*2*pi, β=∞, energy=false, all_modes=true)

In [ ]:
ϕ_t_normal_modes[1]

In [ ]:
length(t)

In [ ]:

n=1000
bath_params=(ω_c=50, ω_cut_off=100)
bath=Exp_decay
AB, J = bath(bath_params=bath_params, κ=κ, C_inv=C_inv)
onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

### Mapping ###
Z_sys = C_inv / ω_sys # Characteristic impedance in GHz
ω = vcat(ω_sys,onsite_energies) # GHz
g_q=vcat(0, -hopping_energies) 
g_ϕ=vcat(-system_chain_coupling*sqrt(Z_sys), hopping_energies) 

ϕ_init = 1.0
q_init = 1.0

ϕ_0 = vcat(SVector(ϕ_init/sqrt(Z_sys)),SVector{n}(zeros(n)))
q_0 = vcat(SVector(q_init*sqrt(Z_sys)),SVector{n}(zeros(n)))

Mq=SymTridiagonal(ω, g_q)
inv_Mq= inv(Mq)
Mϕ=SymTridiagonal(ω, g_ϕ)

A=Mq * Mϕ
vals, vecs = eigen(A)

ωk² = real(vals)
ωk = sqrt.(ωk²)
P = real(vecs)  # columns = normal modes

P_inv=inv(P)

# Project initial conditions into normal mode basis
ϕ_0_normal_modes = (P_inv * ϕ_0)
ϕ_dot_0_normal_modes = (P_inv * Mq * q_0)


tspan = (0.0, t_max)
dt=1/(20*maximum(ωk))
npoints = Int(round((tspan[2]-tspan[1])/dt))  # number of time points

In [ ]:
# Time evolution in normal coordinates
tlist = range(tspan[1], tspan[2], length=npoints)
ϕ_t_normal_modes = [ϕ_0_normal_modes .* cos.(ωk .* t) .+ (ϕ_dot_0_normal_modes ./ ωk) .* sin.(ωk .* t) for t in tlist]
ϕ_t_dot_normal_modes= [ϕ_0_normal_modes .* (-ωk) .* sin.(ωk .* t) .+ (ϕ_dot_0_normal_modes) .* cos.(ωk .* t) for t in tlist]
q_t_normal_modes = [P_inv * inv_Mq * P * a for a in ϕ_t_dot_normal_modes]
# Reconstruct real-space fields
ϕ_t = [P * a for a in ϕ_t_normal_modes]
ϕ_t_dot = [P * a for a in ϕ_t_dot_normal_modes] # \dot{ϕ} = P * \dot{ϕ}_normal_modes
q_t = [inv_Mq * P * a for a in ϕ_t_dot_normal_modes] # q = \dot{ϕ}/Mq

ϕ_sys =  [ϕ_t[i][1] for i in 1:length(tlist)]*sqrt(Z_sys)
q_sys =  [q_t[i][1] for i in 1:length(tlist)]/ sqrt(Z_sys)

In [ ]:
D1= diag(P' * inv_Mq * P)
D2= diag(P' * Mϕ * P);

In [ ]:
Energies_normal_modes = [D1[i]*getindex.(ϕ_t_dot_normal_modes,i)[1]^2 + D2[i]*getindex.(ϕ_t_normal_modes,i)[1]^2 for i in 1:n+1]

In [ ]:
sum(Energies_normal_modes)

In [ ]:
E_sys= q_sys.^2*C_inv + ϕ_sys.^2*ω_sys^2/C_inv

In [ ]:
P'

In [ ]:
P_inv

In [ ]:
plot(ωk/2pi, Energies_normal_modes)
plot!(ωk/2pi, E_sys[1]*P[1,:].^2/sum(P[1,:].^2))
plot!(xlims=(4.5,5.5))

In [ ]:
plot(ωk/2pi, Energies_normal_modes-E_sys[1]*P[1,:].^2/sum(P[1,:].^2))
plot!(ωk/2pi, Energies_normal_modes-E_sys[end]*P[1,:].^2/sum(P[1,:].^2))
plot!(xlims=(4.5,5.5))

In [ ]:
plot(tlist,E_sys[1].-E_sys)

In [ ]:
F, freqs = FFT_perso(E_sys[1].-E_sys,tlist)
plot(freqs, abs.(F),xlims=(-10, 10))

In [ ]:
### Mapping ###
Z_sys = C_inv / ω_sys # Characteristic impedance in GHz
ω_chain = vcat(onsite_energies) # GHz
g_q_chain=vcat(-hopping_energies) 
g_ϕ_chain=vcat(hopping_energies) 

Mq_chain=SymTridiagonal(ω_chain, g_q_chain)
inv_Mq_chain= inv(Mq_chain)
Mϕ_chain=SymTridiagonal(ω_chain, g_ϕ_chain)

A_chain=Mq_chain * Mϕ_chain
vals_chain, vecs_chain = eigen(A_chain)

ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)
P_chain_inv=inv(P_chain)

In [ ]:
ϕ_t_chain = @view ϕ_t[1:npoints]  # shorten if needed
ϕ_t_dot_chain = @view ϕ_t_dot[1:npoints]
# Allocate result
Energies_normal_modes_chain = Matrix{Float64}(undef, n, npoints)

# Compute S_chain and S_chain_dot on the fly, avoid storing large intermediates
for j in 1:npoints
    S = P_chain_inv * @view ϕ_t_chain[j][2:n+1]
    S_dot = P_chain_inv * @view ϕ_t_dot_chain[j][2:n+1]
    for i in 1:n
        Energies_normal_modes_chain[i, j] = D1_chain[i] * S_dot[i]^2 + D2_chain[i] * S[i]^2
    end
end

In [ ]:
Energies_normal_modes_chain[:,1]

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain[:,end], label="")
plot!(ylabel="Energy stored in the chain\n normal modes", xlabel="Frequency (GHz)", xlims=(4.5,5.5), title = "t = $(t_max) ns, n=$(n)")
#savefig("Energies_normal_modes_chain_one_osc.pdf")

## Coupling strength

In [ ]:
## Couplign strength
plot(ωk/2pi, P[1,:].^2/sum(P[1,:].^2), label="")
plot!(xlims=(4.5,5.5), ylabel="Coupling strength", xlabel="Frequency (GHz)")
#savefig("coupling_strength_one_osc.pdf")

# Linear oscillator, inverting the chain mapping

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
ω_sys=5*2*pi  # GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
L = C_inv / ω_sys^2 # L in 1/GHz
Z_sys = sqrt(L*C_inv) # Characteristic impedance (dimensionless)
κ=1/20; # 1/ns from Jeffrey et al.

In [ ]:
n_huge=2000
AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=90))
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 2/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
bath = Exp_decay
bath_params=(ω_c=50, ω_cut_off=100)
ϕ,ϕ_dot,t,Mq_chain,Mϕ_chain= One_d_linear(n, bath, bath_params, t_max; ω_sys=5*2pi, κ=1/20, C_inv=0.147*2*pi, β=∞, emission=true);

In [ ]:
ϕ_chain=[getindex.(ϕ,i) for i in 2:n+1]
ϕ_dot_chain=[getindex.(ϕ_dot,i) for i in 2:n+1];

In [ ]:
ωk_chain,Energies_normal_modes_chain = emission_spectrum(ϕ_chain, ϕ_dot_chain, Mϕ_chain, Mq_chain)

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain[end,:])
vline!([ω_sys/2pi],linestyle=:dash, color=:red)
annotate!(ω_sys/2pi+.05,1e-3, Plots.text("system mode", 8, :red, rotation = 90 ))
plot!(ylabel="Energy stored in the chain", 
xlabel="Frequency (GHz)", 
title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", 
legend=false,
xlims=(4,6),
ylims=(1e-5, 1e3),
yaxis=:log)
#savefig("../Figures/Emission_spectrum/Energies_normal_modes_chain_one_osc_log.pdf")